**DATA CLEANING NOTES PER DATASET**

1. Gemini Dataset
    - A bit easy to clean if we were to base the cleaning on the

2. Claude Dataset
    - Needs a bit of cleaning since it has a lot of curly braces
    - It has equations and code 
    - to clean: remaining greek operators, fractions, and code

3. MGTBench Dataset (ChatGPT)
    - heavy in data cleaning especially for subjects in physics
    - has new lines that needs to be removed

4. MGTBench Dataset (Human-written)
    - heavy in data cleaning especially for subjects in physics
    - has new lines that needs to be removed

5. BAWE Corpus Dataset
    - Needs the most cleaning since it an HTML file 
    - Need to convert it into an HTML file so we can clean it properly

In [1]:
# Import the necessary libraries for cleaning the data
import os
import re
import pandas as pd
import numpy as numpy
from pathlib import Path
from tqdm import tqdm
import ftfy
import ast

pd.set_option('display.max_colwidth', 150)
tqdm.pandas(desc="Cleaning Text")

print("Libraries has been imported!")

Libraries has been imported!


In [2]:
# Establish the directories where the data will be read and stored after processing
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "data").exists() else CURRENT_DIR.parent.parent
DATA_DIR = PROJECT_ROOT / "data"

# Directories of the Raw Datasets
RAW_AI_DIR = DATA_DIR / "raw" / "ai"
RAW_HUMAN_DIR = DATA_DIR / "raw" / "human"

# Directories of the Processed Datasets where it will be stored after cleaning the data
PROCESSED_AI_DIR = DATA_DIR / "processed" / "ai"
PROCESSED_HUMAN_DIR = DATA_DIR / "processed" / "human"

# To ensure that the out processed directories exist
PROCESSED_AI_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_HUMAN_DIR.mkdir(parents=True, exist_ok=True)

print("Data Cleaning Paths Ready:")
print(f"  Reading AI Raw Data:        {RAW_AI_DIR.resolve()}")
print(f"  Reading Human Raw Data:     {RAW_HUMAN_DIR.resolve()}")
print(f"  Saving AI Processed Data:   {PROCESSED_AI_DIR.resolve()}")
print(f"  Saving Human Processed Data:{PROCESSED_HUMAN_DIR.resolve()}")

Data Cleaning Paths Ready:
  Reading AI Raw Data:        C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\raw\ai
  Reading Human Raw Data:     C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\raw\human
  Saving AI Processed Data:   C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai
  Saving Human Processed Data:C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\human


**METHODS FOR CLEANING DATA**

- Cleans math texts
- Cleans texts that has citations
- Cleans texts that has code in it
- Strips reference lists
- Cleans texts that has any numberings in it

In [8]:
def clean_math_texts(text):
    """
    Cleans math equations, LaTeX expressions, and Unicode mathematical symbols from text
    by replacing them with <EQUATION> and <SYMBOL> placeholders.
    """
    if not isinstance(text, str):
        return text

    # Normalize literal escaped newlines and citation markers
    text = text.replace('\\n', ' ')
    text = re.sub(r'\[\d+\]', '', text)

    # LaTeX environments
    text = re.sub(r'\\begin\{[a-zA-Z0-9\*]+\}.*?\\end\{[a-zA-Z0-9\*]+\}', ' <EQUATION> ', text, flags=re.DOTALL)

    # LaTeX block and display math
    text = re.sub(r'\$\$.*?\$\$', ' <EQUATION> ', text, flags=re.DOTALL)
    text = re.sub(r'\\\[.*?\\\]', ' <EQUATION> ', text, flags=re.DOTALL)

    # LaTeX inline math
    text = re.sub(r'\$([^\$\n]+)\$', ' <EQUATION> ', text)
    text = re.sub(r'\\\((.*?)\\\)', ' <EQUATION> ', text)

    # Common LaTeX commands
    text = re.sub(r'\\[a-zA-Z]+(\{.*?\})*', ' <EQUATION> ', text)

    # Integrals with bounds and differentials
    text = re.sub(r'[∮∫][₀-₉⁰-⁹\^]*[^\.\n]*?d[A-Za-z]+', ' <EQUATION> ', text)

    # Algebraic equations containing '=' or comparison operators
    text = re.sub(r'\b[a-zA-Z0-9\(\)\{\}\[\]\+\-\*/\^]+\s*(?:<=|>=|!=|==|=|<|>|≠|≤|≥|≈)\s*[a-zA-Z0-9\(\)\{\}\[\]\+\-\*/\^\s\._]+', ' <EQUATION> ', text)

    # Exponents and derivatives
    text = re.sub(r'\b[a-zA-Z0-9\(\)]+\^[a-zA-Z0-9\(\)\+\-]+\b', ' <EQUATION> ', text)
    text = re.sub(r'\bd[A-Za-z]/d[A-Za-z]\b', ' <EQUATION> ', text)

    # "n choose k" style combinatorics notation
    text = re.sub(r'\([a-zA-Z0-9\s\+\-]+choose[a-zA-Z0-9\s\+\-]+\)', '<EQUATION>', text)

    # Catches sentences that survived token-level cleaning but are still mostly fragments/placeholders
    cleaned_sentences = []
    sentences = re.split(r'(?<=[.!?])\s+', text)
    prev_was_equation = False

    for sent in sentences:
        placeholder_count = sent.count('<EQUATION>')
        non_placeholder_text = re.sub(r'<EQUATION>', '', text)
        word_count = len(re.findall(r'[a-zA-Z]{3,}', non_placeholder_text))

        # If a ssentence has 2+ placeholders and very few real words around them,
        # it's a fragment soup, so we need to collapise to a single <EQUATION>
        is_fragment_heavy = placeholder_count >= 2 and word_count < 6

        if is_fragment_heavy or (placeholder_count >= 1 and word_count == 0):
            if not prev_was_equation:
                cleaned_sentences.append("<EQUATION>")
            prev_was_equation = True
        else:
            cleaned_sentences.append(sent)
            prev_was_equation = False

    text = ' '.join(cleaned_sentences)
    text = re.sub(r'(<EQUATION>\s*){2,}', '<EQUATION>', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def strip_reference_list(text):
    """Cuts off everything from the first 'Sources:' / 'References' """

    match = re.search(r'\n?(Sources|References|Bibliography):', text, flags=re.IGNORECASE)

    if match:
        return text[:match.start()].strip()

    return text

def extract_claude_text(text):
    """
        Extracts the text content from stringified conversation list dictionaries 
        [{'from': 'human', 'value': ...}, {'from': 'gpt', 'value': ...}]
    """

    if not isinstance(text, str):
        return text

    # Checks if the text looks like a list of conversation dictionaries
    if text.strip().startswith('[') and "'from'" in text and "'value'" in text:
        try:
            data = ast.literal_eval(text)
            claude_turns = [turn['value'] for turn in data if isinstance(turn, dict) and turn.get('from') in ('gpt', 'assistant')]

            if claude_turns:
                return " ".join(claude_turns)
        except Exception:
            # Fallback regex cleanup if ast parsing encounters malformed formatting
            text = re.sub(r"\[?\s*\{'from':\s*'[a-zA-Z]+',\s*'value':\s*", "", text)
            text = re.sub(r"'\}\s*\,?\s*\]?", "", text)
    
    return text


def clean_code_texts(text):
    """
        Cleans code blocks and inline code snippets from text by replacing them 
        with <CODE> placeholders.
    """

    if not isinstance(text, str):
        return text

    # Extract the texts that are in dictionary style
    text = extract_claude_text(text)

    # Replace markdown fenced code blocks
    text = re.sub(r'```[a-zA-Z0-9_\-\+\#]*\n?.*?\n?```', ' <CODE> ', text, flags=re.DOTALL)

    # Replace inline code snippets
    text = re.sub(r'`[^`\n]+`', ' <CODE> ', text)

    # Consolidate consecutive <CODE> tags and normalize whitespace
    text = re.sub(r'(<CODE>\s*){2,}', '<CODE> ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_citations(text):
    """
        Replaces any citations whether intext to <CITATION>
    """
    if not isinstance(text, str):
        return text

    # Bracketed numeric citations: [1], [12], [1,2]
    text = re.sub(r'\[\d+(,\s*\d+)*\]', ' <CITATION> ', text)

    # Parenthetical author-date citations: (Smith, 2020), (Griffiths, D. J., 2017)
    text = re.sub(r'\([A-Z][a-zA-Z\.\s,&]+,\s*\d{4}[a-z]?\)', ' <CITATION> ', text)

    # Narrative/paraphrased citations: "Smith (2020)", "Griffiths (2017)"
    text = re.sub(r'\b[A-Z][a-zA-Z]+\s\(\d{4}[a-z]?\)', ' <CITATION> ', text)

    # Cleanup
    text = re.sub(r'(<CITATION>\s*){2,}', '<CITATION> ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def clean_list_numbering(text):
    """
        Cleans texts that has any numbering in it
        e.g. 1. 2. ...
    """

    if not isinstance(text, str):
        return text

    # Numbered list markers at start of a segment
    text = re.sub(r'(?:^|\n)\s*\d+\.\s*', ' ', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [9]:
from IPython.display import display

# Combined pipeline function that applies all cleaning methods
def clean_pipeline(text):
    text = clean_code_texts(text)      # Extracts JSON conversation text & cleans code
    text = clean_math_texts(text)      # Cleans math equations & Unicode symbols
    text = clean_citations(text)       # Cleans bracketed & author-date citations
    text = clean_list_numbering(text)  # Removes list numbering (e.g. 1., 2.)
    text = strip_reference_list(text)  # Strips reference lists at end
    return text

claude_path = RAW_AI_DIR / "claude_dataset.csv"

if claude_path.exists():
    print(f"Loading first 500 rows from {claude_path.name}...")
    df_claude = pd.read_csv(claude_path, nrows=500)
    print("Applying all cleaning methods...")
    df_claude['cleaned_text'] = df_claude['conversations'].progress_apply(clean_pipeline)

    # Remove raw 'conversations' column, leaving only 'cleaned_text'
    df_claude = df_claude[['cleaned_text']]
    
    # 1. Summary Statistics Table
    summary_data = {
        "Metric": [
            "Total Rows Processed", 
            "<EQUATION> Tags Inserted", 
            "<SYMBOL> Tags Inserted", 
            "<CODE> Tags Inserted",
            "<CITATION> Tags Inserted"
        ],
        "Count": [
            len(df_claude),
            df_claude['cleaned_text'].str.count('<EQUATION>').sum(),
            df_claude['cleaned_text'].str.count('<SYMBOL>').sum(),
            df_claude['cleaned_text'].str.count('<CODE>').sum(),
            df_claude['cleaned_text'].str.count('<CITATION>').sum()
        ]
    }
    df_summary = pd.DataFrame(summary_data)
    
    print("\n--- CLEANING SUMMARY ---")
    display(df_summary)
    # 2. Save the 500 cleaned rows (cleaned_text only) to PROCESSED_AI_DIR
    output_path = PROCESSED_AI_DIR / "claude_dataset_cleaned_500.csv"
    df_claude.to_csv(output_path, index=False)
    print(f"\nSuccessfully saved 500 cleaned rows to: {output_path}")
    # 3. Display the first 20 cleaned rows table
    print("\n--- FIRST 20 PROCESSED ROWS ---")
    display(df_claude.head(20))
else:
    print(f"File not found at: {claude_path}")

Loading first 500 rows from claude_dataset.csv...
Applying all cleaning methods...


Cleaning Text:   0%|          | 0/500 [00:00<?, ?it/s]

Cleaning Text: 100%|██████████| 500/500 [00:02<00:00, 240.93it/s]


--- CLEANING SUMMARY ---


,Metric,Count
0,Total Rows Processed,500
1,<EQUATION> Tags Inserted,927
2,<SYMBOL> Tags Inserted,0
3,<CODE> Tags Inserted,0
4,<CITATION> Tags Inserted,113



Successfully saved 500 cleaned rows to: c:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai\claude_dataset_cleaned_500.csv

--- FIRST 20 PROCESSED ROWS ---


,cleaned_text
0,The binomial theorem states that for any real numbers x and y and a non-negative integer n: (x + <EQUATION> ∑ <EQUATION> −k) <EQUATION> Where <EQU...
1,"The process of manufacturing a violin is an intricate and time-consuming endeavor that requires a high level of skill, precision, and artistry. He..."
2,Thank you for providing such a comprehensive approach to debugging and improving code quality. This is an excellent methodology that covers many i...
3,"Here's a Python function that counts the number of vowels in a given string: < <EQUATION> : 1. We define a set of vowels, including both lowercase..."
4,"To effectively structure one's writing, it's important to employ clear topic sentences, smooth transitions, a consistent theme, and proper punctua..."
5,Percutaneous Coronary Intervention (PCI) is a minimally invasive procedure used to treat coronary artery disease. Here's a detailed breakdown of t...
6,Here's a comprehensive overview of algorithms and data structures commonly used in software engineering: 1. Searching Algorithms a) Linear Search ...
7,Certainly! I'll provide a proof for the Pythagorean Theorem and then discuss how to help students understand and remember this important mathemati...
8,"an imaginative story poem depicting a revolutionary invention, including the process of invention, testing, and breakthrough moment, expressing th..."
9,"Faraday's Experiments and Magnetic Flux: In the 1830s, Michael Faraday conducted a series of experiments that established the fundamental principl..."


In [ ]:
def clean_claude_dataset(dataset):
    
    pass

In [ ]:
def clean_mgtbench_human_dataset(dataset):

    pass

In [ ]:
def clean_mgtbench_ai_dataset(dataset):

    pass

In [ ]:
def clean_bawe_corpus_dataset(dataset):

    pass